In [41]:
import pm4py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load the XES file
log = pm4py.read_xes("BPI Challenge 2017.xes")

# Convert the log to a DataFrame
log_df = pm4py.convert_to_dataframe(log)

In [ ]:
# Sneak peek
print(log_df.shape)             
print(log_df.columns.tolist()) 
log_df.head()                   

In [ ]:
## Basic statistics

case_id = "case:concept:name"

# Number of cases and events
n_cases = log_df[case_id].nunique()
n_events = len(log_df)

# Number of variants (unique activity sequences) and distinct attribute labels
n_variants = len(pm4py.get_variants(log_df))
case_attr_labels = pm4py.get_trace_attributes(log_df)   # trace/case-level attribute names
event_attr_labels = pm4py.get_event_attributes(log_df)  # event-level attribute names

print(f"Number of events: {n_events}")
print(f"Number of cases: {n_cases}")
print(f"Number of variants: {n_variants}")
print(f"Number of distinct case attribute labels: {len(case_attr_labels)}")
print(f"Number of distinct event attribute labels: {len(event_attr_labels)}")


In [ ]:
## Case Length Statistics (events per case)

# Mean and standard deviation of case length
case_lengths = log_df.groupby(case_id).size()
mean_case_length = case_lengths.mean() # mean number of events per case
std_case_length  = case_lengths.std() # standard deviation of number of events per case

print(f"Case length (events/case):  mean={mean_case_length:.2f}, std={std_case_length:.2f}")

In [ ]:
## Case Duration Statistics

# Mean and standard deviation of case duration (time from first to last event per case)
case_dur_sec = pm4py.get_all_case_durations(log_df)

mean_dur_sec  = np.mean(case_dur_sec)
std_dur_sec   = np.std(case_dur_sec, ddof=1)  # ddof=1 for sample std, default for pandas

print(f"Case duration (days):    mean={mean_dur_sec/86400:.2f}, std={std_dur_sec/86400:.2f}")
print(f"Case duration (minutes): mean={mean_dur_sec/60:.1f}, std={std_dur_sec/60:.1f}")
print(f"Case duration (seconds): mean={mean_dur_sec:.0f}, std={std_dur_sec:.0f}")

In [ ]:
## Categorical Event Attributes

# Number of categorical event attributes (with and without case)
event_attrs = [c for c in pm4py.get_event_attributes(log_df) if not c.startswith("case:")]

categorical_event_attrs = (
    log_df[event_attrs]
    .select_dtypes(include=["object", "category", "bool"])
    .columns
    .tolist()
)

print(f"Number of categorical event attributes: {len(categorical_event_attrs)}")
print("Categorical event attributes:", categorical_event_attrs)

In [ ]:
## Case length distribution (events per case)

# Case length: events/case
case_lengths = log_df.groupby(case_id).size()
median_case_length = case_lengths.median()
longest_case_length = case_lengths.max()

# Bins with width 2 events up to 120, overflow bin >=120
upper_limit_len = 120
bin_width_len = 2

case_lengths_plot = case_lengths.clip(upper=upper_limit_len + bin_width_len)
bin_edges_len = list(range(0, upper_limit_len + bin_width_len + 1, bin_width_len))

plt.figure(figsize=(9, 5))
plt.hist(case_lengths_plot, bins=bin_edges_len, edgecolor="black")
plt.axvline(median_case_length, color="red", linestyle="--", linewidth=2,
            label=f"Median = {median_case_length:.2f} events")

# Plot longest case length in legend
plt.plot([], [], ' ', label=f"Longest case = {longest_case_length} events")

plt.xlim(0, upper_limit_len + bin_width_len)
plt.xticks([0, 20, 40, 60, 80, 100, 120],
           ["0", "20", "40", "60", "80", "100", ">=120"])

plt.title("Case Length Distribution")
plt.xlabel("Case length (number of events)")
plt.ylabel("Number of cases")
plt.grid(axis="y", color="lightgrey", linestyle="-", linewidth=0.6, alpha=0.7)
plt.legend()
plt.show()

In [ ]:
## Case length histogram and case duration histogram + medians

# Case duration in seconds
case_dur_sec = pd.Series(pm4py.get_all_case_durations(log_df))

# Convert to days for plotting
case_dur_days = case_dur_sec / 86400
median_case_dur_days = case_dur_days.median()

# Longest case duration in days
longest_case_dur_days = case_dur_days.max()
print(f"Longest case duration: {longest_case_dur_days:.2f} days")

# Bins with length 2 days up to 80 and one overflow bin >80
upper_limit = 80
bin_width = 2

case_dur_days_plot = case_dur_days.clip(upper=upper_limit + bin_width)
bin_edges = list(range(0, upper_limit + bin_width + 1, bin_width))

plt.figure(figsize=(9, 5))
plt.hist(case_dur_days_plot, bins=bin_edges, edgecolor="black")
plt.axvline(median_case_dur_days, color="red", linestyle="--", linewidth=2,
            label=f"Median = {median_case_dur_days:.2f} days")

# Plot longest case duration in legend
plt.plot([], [], ' ', label=f"Longest case = {longest_case_dur_days:.2f} days")

plt.xlim(0, upper_limit + bin_width)
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80],
           ["0", "10", "20", "30", "40", "50", "60", "70", ">=80"])

plt.title("Case Duration Distribution")
plt.xlabel("Case duration (days)")
plt.ylabel("Number of cases")
plt.grid(axis="y", color="lightgrey", linestyle="-", linewidth=0.6, alpha=0.7)
plt.legend()
plt.show()